In [1]:
import numpy as np

# Timeseries Data Preparation

## 1 Utility Functions

### `generator` function from Notebook 2B

In [2]:
def generator(
    data, target_col, lookback, delay, step, min_index, max_index, shuffle=False,
    batch_size=128):
    """Generate the sample and target from the given timeseries data

    Parameters
    ----------
    data : array-like
        Timeseries data with all the features to be used as predictors.
        Has an expected shape of num_rows x num_features.

    target_col : int
        Index of the column in `data` which would be used as the target

    lookback : int
        How far back in history do we use to construct the sample

    delay : int
        Time difference between target and sample. A zero delay
        corresponds to the next timestep wrt the sample.

    step : int
        Time step between values within the sample

    min_index : int
        Minimum index to be considered when constructing the sample.
        Useful for separating train, validation, and test datasets.

    max_index : int
        Maximum index to be considered when constructing the sample.
        Useful for separating train, validation, and test datasets.

    shuffle : bool
        Whether to shuffle the resulting sample and target pairs.

    batch_size : int
        The batch size

    Yields
    ------
    samples, target : array-like
        For each iteration, this function yields the samples and target
        pairs to be used as predictors and targets. Samples has a shape
        of `batch_size` x `num_features` meanwhile target has a shape of
        `batch_size`
    """

    # Set default max_index if not provided
    if max_index is None:
        max_index = len(data) - delay - 1

    # Start index for data sampling
    i = min_index + lookback

    # Infinite loop to generate samples indefinitely
    while True:
        # Shuffle data if required; otherwise, generate sequential batches
        if shuffle:
            rows = np.random.randint(min_index + lookback, max_index, size=batch_size)
        else:
            if i + batch_size >= max_index:
                i = min_index + lookback  # Reset index if it goes beyond the max index
            rows = np.arange(i, min(i + batch_size, max_index))
            i += len(rows)

        # Initialize numpy arrays to store samples and targets
        samples = np.zeros((len(rows), lookback // step, data.shape[-1]))
        targets = np.zeros((len(rows),))

        # Populate samples and targets arrays
        for j, row in enumerate(rows):
            indices = range(rows[j] - lookback, rows[j], step)  # Select indices for sample
            samples[j] = data[indices]  # Assign the sliced data to samples
            targets[j] = data[rows[j] + delay][target_col]  # Set the corresponding target

        # Yield a batch of samples and targets
        yield samples, targets

In [3]:
help(generator)

Help on function generator in module __main__:

generator(data, target_col, lookback, delay, step, min_index, max_index, shuffle=False, batch_size=128)
    Generate the sample and target from the given timeseries data
    
    Parameters
    ----------
    data : array-like
        Timeseries data with all the features to be used as predictors.
        Has an expected shape of num_rows x num_features.
    
    target_col : int
        Index of the column in `data` which would be used as the target
    
    lookback : int
        How far back in history do we use to construct the sample
    
    delay : int
        Time difference between target and sample. A zero delay
        corresponds to the next timestep wrt the sample.
    
    step : int
        Time step between values within the sample
    
    min_index : int
        Minimum index to be considered when constructing the sample.
        Useful for separating train, validation, and test datasets.
    
    max_index : int
       

### From Keras Utils `timeseries_dataset_from_array()`

In [4]:
from tensorflow import keras

In [5]:
help(keras.utils.timeseries_dataset_from_array)

Help on function timeseries_dataset_from_array in module keras.src.utils.timeseries_dataset:

timeseries_dataset_from_array(data, targets, sequence_length, sequence_stride=1, sampling_rate=1, batch_size=128, shuffle=False, seed=None, start_index=None, end_index=None)
    Creates a dataset of sliding windows over a timeseries provided as array.
    
    This function takes in a sequence of data-points gathered at
    equal intervals, along with time series parameters such as
    length of the sequences/windows, spacing between two sequence/windows, etc.,
    to produce batches of timeseries inputs and targets.
    
    Args:
      data: Numpy array or eager tensor
        containing consecutive data points (timesteps).
        Axis 0 is expected to be the time dimension.
      targets: Targets corresponding to timesteps in `data`.
        `targets[i]` should be the target
        corresponding to the window that starts at index `i`
        (see example 2 below).
        Pass None if you

## 2 Examples Using `numpy` Indices

In [6]:
data = np.arange(0, 15, 1)[:, np.newaxis]
data

array([[ 0],
       [ 1],
       [ 2],
       [ 3],
       [ 4],
       [ 5],
       [ 6],
       [ 7],
       [ 8],
       [ 9],
       [10],
       [11],
       [12],
       [13],
       [14]])

In [7]:
data.shape

(15, 1)

### `generator`

**Parameter Settings**

```
Lookback: 7
Delay: 4
Step: 1
```

**Parameter Settings**

```
Lookback: 6
Delay: 3
Step: 2
```

### `timeseries_dataset_from_array()`

**Parameter Settings**

```
Ahead: 5
Sequence Length: 7
Sampling Rate: 1
Sequence Stride: 1
```

**Parameter Settings**

```
Ahead: 2.5
Sequence Length: 3
Sampling Rate: 2
Sequence Stride: 1
```

**Parameter Settings**

```
Ahead: 4
Sequence Length: 3
Sampling Rate: 2
Sequence Stride: 1
```

**Parameter Settings**

```
Ahead: 4
Sequence Length: 3
Sampling Rate: 2
Sequence Stride: 2
```

## 3 Examples Using Hourly Energy Consumption

In [24]:
from glob import glob

import pandas as pd

### Data Loading and Preprocessing

### Scenario 1

Use the hourly energy consumption of `AEP`, `PJMW`, and `PJME` to predict the total daily consumption of `PJMW` one month in advance using 2 weeks worth of hourly consumption data.

#### Using the `generator` function

#### Using `timeseries_dataset_from_array()`

### Scenario 2

Use the 6-hourly energy consumption of `AEP`, `PJMW`, and `PJME` to predict the total daily consumption of `PJMW` one month in advance using one month worth of hourly consumption data.

#### Using the `generator` function

#### Using `timeseries_dataset_from_array()`